In [2]:
from mlxtend import classifier
from sklearn.base  import BaseEstimator , ClassifierMixin
import numpy as np
from sklearn.utils import check_X_y
from statsmodels.tsa.base import prediction

# Custom Estimator

In [3]:
class MostFrequentClassClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self):
        self.most_frequent_ = None

    def fit(self,X,y):

        # Validate input X and target vector y
        X,y = check_X_y(X,y)

        # Ensure y in 1D
        y = np.ravel(y)

        # Manually compute the most frequent class
        unique_classes, counts =np.unique(y, return_counts=True)
        self.most_frequent_ = unique_classes[np.argmax(counts)]

        return self

    def predict(self,X):
        if self.most_frequent_ is None:
            raise ValueError("This classifier instance has not been fitted yet.")
        # Predict the most frequent class for each input sample
        return np.full(shape=(X.shape[0],), fill_value=self.most_frequent_)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

# Load data
iris = load_iris()
X, y = iris.data, iris.target

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and fit the custom estimator
classifier = MostFrequentClassClassifier()
classifier.fit(X_train, y_train)

# Make prediction
predictions = classifier.predict(X_test)

# Evaluate the custom estimator
print(f"Predicted class for all test instances: {predictions[0]}")

Predicted class for all test instances: 1


In [5]:
classifier.most_frequent_

1

In [6]:
from sklearn.model_selection import cross_val_score
cross_val_score(classifier, X_train, y_train)

array([0.33333333, 0.33333333, 0.29166667, 0.25      , 0.20833333])

# Scoring Function

In [8]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score

class MostFrequentClassClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self):
        self.most_frequent_ = None

    def fit(self,X,y):
        y = np.ravel(y)

        # computing most frequent
        unique_classes, counts = np.unique(y,return_counts=True)
        self.most_frequent_ = unique_classes[np.argmax(counts)]

    def predict(self, X):
        if self.most_frequent_ is None:
            raise ValueError("This classifier instance has not been fitted yet.")
        return np.full(shape=(X.shape[0],), fill_value=self.most_frequent_)

    def score(self, X, y):
        y = np.ravel(y)
        predictions = self.predict(X)
        return accuracy_score(y, predictions)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

# Load a dataset
iris = load_iris()
X, y = iris.data, iris.target

# Simplify to a binary classification problem
is_class_0_or_1 = y < 2
X_bin = X[is_class_0_or_1]
y_bin = y[is_class_0_or_1]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_bin, y_bin, test_size=0.2, random_state=42)

# Initialize and fit the custom classifier
classifier = MostFrequentClassClassifier()
classifier.fit(X_train, y_train)

# Evaluate the classifier using the score method
score = classifier.score(X_test, y_test)
print(f"Accuracy of the MostFrequentClassClassifier: {score}")


Accuracy of the MostFrequentClassClassifier: 0.4


# Transformers

In [10]:
from sklearn.datasets import make_regression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# Generate Some Data
X,y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)

# Use the transformer directly
X_transformed = StandardScaler().fit_transform(X)

LinearRegression().fit(X_transformed, y)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


## Custom Transformer using Function Transformer

In [11]:
import numpy as np

def cube(x):
    return np.power(x,3)

In [12]:
from sklearn.preprocessing import FunctionTransformer

# creating custom transformer
cube_transformer = FunctionTransformer(cube)

In [13]:
from sklearn.datasets import make_regression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# Generate Some Data
X,y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)

# Use the transformer directly
X_transformed = cube_transformer.transform(X)

LinearRegression().fit(X_transformed, y)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False



## Custom Transformer using BaseEstimator and TransformerMixin

In [14]:
from sklearn.base import BaseEstimator,TransformerMixin
import numpy as np

In [15]:
class MedianIQRScaler(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.medians_ = None
        self.iqr_ = None

    def fit(self, X, y=None):
        # calculating medians and interquartile range for each feature
        self.medians_ = np.median(X, axis=0)
        Q1 = np.percentile(X,25,axis=0)
        Q3 = np.percentile(X,75,axis=0)
        self.iqr_ = Q3-Q1

        # Handle case where IQR is 0 to avoid division by zero during transform
        self.iqr_[self.iqr_ == 0] = 1
        return self

    def transform(self, X):
        # Check if fit has been called
        if self.medians_ is None or self.iqr_ is None:
            raise ValueError("This estimator has not been fitted yet.")
        # scale features
        return (X-self.medians_)/self.iqr_

In [16]:
from sklearn.datasets import make_blobs

# Generate synthetic data
X, _ = make_blobs(n_samples=100, n_features=2, centers=3, random_state=42)

# Initialize the transformer
scaler = MedianIQRScaler()

# Fit the scaler to the data
scaler.fit(X)

# Transform the data
X_scaled = scaler.transform(X)

# Check the first few rows of the transformed data
print("Transformed data (first 5 rows):")
print(X_scaled[:5])


Transformed data (first 5 rows):
[[-0.49872679 -0.71613207]
 [ 0.78423675 -0.08192868]
 [-0.03656645  0.52987512]
 [ 0.84159877 -0.09379661]
 [-0.3814692  -0.57206564]]
